## 4b. [KIỂM TRA TRƯỚC KHI CHẠY FULL] Xác minh trực quan 3 ảnh từng có bug

Chạy đúng pipeline thật (padding + grayscale-check + white balance + CodeFormer) trên 3 ảnh
từng phát hiện bug trước đây, hiển thị ảnh gốc / sau xử lý cạnh nhau để xác nhận bằng mắt
TRƯỚC KHI chạy batch full 1002 ảnh (tốn nhiều giờ CodeFormer).

- `047A05`: kỳ vọng KHÔNG còn hoa văn hình thoi (diamond pattern) quanh viền mặt
- `003A35`, `004A37`: kỳ vọng màu da tự nhiên, KHÔNG ngả xanh lục/cyan, và đặc biệt phải
  xác nhận `is_effectively_grayscale()` KHÔNG chấm nhầm 2 ảnh này là ảnh xám (nếu nhầm,
  White Balance sẽ bị bỏ qua hoàn toàn và màu sepia gốc sẽ không được sửa)

In [ ]:
import matplotlib.pyplot as plt

KNOWN_BUG_FILES = ["047A05.JPG", "003A35.JPG", "004A37.JPG"]

fig, axes = plt.subplots(len(KNOWN_BUG_FILES), 2, figsize=(10, 5 * len(KNOWN_BUG_FILES)))

for row, fname in enumerate(KNOWN_BUG_FILES):
    src_path = os.path.join(FGNET_DIR, fname)
    if not os.path.exists(src_path):
        # Thử tìm với đuôi file khác (.jpg thường/hoa) hoặc thiếu số 0 đầu
        candidates = [f for f in all_images if fname.split('.')[0].upper() in f.upper()]
        if candidates:
            fname = candidates[0]
            src_path = os.path.join(FGNET_DIR, fname)
        else:
            print(f"⚠️ KHÔNG TÌM THẤY: {fname} — kiểm tra lại tên file trong dataset")
            continue

    orig_bgr = cv2.imread(src_path)
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)

    # Chạy đúng luồng thật, từng bước, in rõ quyết định của từng bước
    padded_rgb, was_padded = apply_adaptive_padding(
        orig_rgb, face_occupancy_thresh=0.85, pad_ratio=0.20,
        border_mode="replicate", embedder=app_insight
    )
    is_gray = is_effectively_grayscale(padded_rgb, threshold=6.0)

    if is_gray:
        wb_rgb = padded_rgb
        wb_note = "⚠️ BỊ CHẤM LÀ ẢNH XÁM → BỎ QUA WHITE BALANCE"
    else:
        wb_rgb = apply_white_balance(padded_rgb, p=6.0, max_shift_thresh=35.0)
        wb_note = "Đã chạy Shades of Gray WB (p=6)"

    try:
        final_rgb = run_codeformer(
            wb_rgb, fidelity_weight=0.7,
            unique_tag=f"sanity_{fname.split('.')[0]}",
            temp_dir=TEMP_CODEFORMER_DIR
        )
        cf_note = "CodeFormer OK"
    except Exception as e:
        final_rgb = wb_rgb
        cf_note = f"CodeFormer LỖI (dùng ảnh trước CF): {e}"

    print(f"[{fname}] padded={was_padded} | grayscale={is_gray} | {wb_note} | {cf_note}")

    axes[row, 0].imshow(orig_rgb)
    axes[row, 0].set_title(f"{fname} — GỐC")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(final_rgb)
    axes[row, 1].set_title(f"{fname} — SAU XỬ LÝ")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/sanity_check_known_bugs.png", dpi=150)
plt.show()

print("\n" + "=" * 80)
print("KIỂM TRA BẰNG MẮT:")
print("  - 047A05: có còn hoa văn hình thoi quanh viền mặt không?")
print("  - 003A35, 004A37: màu da có tự nhiên không, hay vẫn ngả xanh lục/cyan?")
print("  - Nếu dòng log có 'BỊ CHẤM LÀ ẢNH XÁM': kiểm tra kỹ màu ảnh SAU XỬ LÝ có còn")
print("    sepia/ngả vàng như ảnh GỐC không — nếu còn, ngưỡng threshold=6.0 đang sai,")
print("    cần hạ ngưỡng xuống hoặc bỏ hẳn bước is_effectively_grayscale cho 2 ảnh này.")
print("CHỈ CHẠY BATCH FULL 1002 ẢNH SAU KHI XÁC NHẬN CẢ 3 ẢNH TRÊN ĐÚNG NHƯ KỲ VỌNG.")
print("=" * 80)